
# Exposure Pipeline (Pythia/OLMo) — Notebook Demo

This notebook demonstrates how to compute **cumulative n‑gram counts** at Pythia (and OLMo) checkpoints and then derive the **exposure feature** per neuron:

![equation](https://latex.codecogs.com/png.latex?\dpi{150}\text{exposure}_j(t)=\log\Big(\max_{g\in\text{Top-}k_j}\text{count}(g;\le%20T_t)+\epsilon\Big))

It uses your standalone library with:
- `NGramIndex` interface and `batch_counts` utility
- Pythia cutoff helpers (`canonical_pythia_cutoffs`)
- A pluggable backend (Tokengrams if available; otherwise an in‑memory demo)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from ngrams.interface import batch_counts
from ngrams.parsers.pythia_cutoffs import canonical_pythia_cutoffs, PYTHIA_TOKENS_PER_STEP

use_tokengrams = False
try:
    from ngrams.backends.tokengrams import TokengramsIndex
    import tokengrams
    use_tokengrams = True
except Exception:
    from ngrams.backends.inmemory import InMemoryIndex

print("Tokengrams available:", use_tokengrams)


## Build or Load the n‑gram Index

- **Tokengrams**: point to the token corpus (binary of token IDs) and an index path.  
- **InMemory** (demo): feed a small token list to get started.


In [ ]:
if use_tokengrams:
    # === Tokengrams path ===
    # Configure these paths to your preshuffled Pythia training token stream.
    # `corpus_path` is a binary file of token IDs (uint16/uint32 depending on vocab),
    # `index_path` is where the tokengrams index will be built/loaded.
    corpus_path = os.environ.get("TOKENGRAMS_CORPUS", "/path/to/preserialized_tokens.bin")
    index_path  = os.environ.get("TOKENGRAMS_INDEX",  "/path/to/preserialized_tokens.idx")
    vocab_size  = int(os.environ.get("VOCAB_SIZE", "50257"))  # set to Pythia's tokenizer vocab

    tg = TokengramsIndex()
    # If you've already built the index at index_path, set load_only=True
    tg.build_index(corpus_path=corpus_path, index_path=index_path, vocab=vocab_size, verbose=True)
    index = tg
else:
    # === In-memory demo path ===
    from src.ngram_counter.backends.inmemory import InMemoryIndex
    demo_tokens = [1, 2, 1, 2, 3, 1, 4, 1, 2, 3, 1]
    idx = InMemoryIndex()
    idx.build_index(demo_tokens, max_n=3)
    index = idx

index



## Pythia Checkpoint Cutoffs

The canonical Pythia schedule (docs): **2,097,152 tokens per step**.  
We’ll compute `(steps, cutoffs)` and then **adjust** cutoffs for `batch_counts` by converting token counts to **inclusive token indices** (subtract 1).


In [ ]:

steps, cutoffs_tokens = canonical_pythia_cutoffs(include_early=True)
# batch_counts expects cutoffs as inclusive last index in the stream, not token counts.
# Convert token counts T to inclusive index by T-1 (and clip at >=0).
cutoffs = np.maximum(cutoffs_tokens - 1, 0)

print("num checkpoints:", len(steps))
print("first few steps:", steps[:10])
print("first few cutoffs (token counts):", cutoffs_tokens[:10])



## Neuron Triggers (Placeholder)

Provide `neuron_to_triggers` as a mapping from neuron id to a **set of token‑ID n‑grams** (length 1–3) discovered from activation snapshots.

For this demo, we’ll stub a few n‑grams. Replace with your actual triggers.


In [ ]:

# Example format: { neuron_id: [ (token_id,), (token_id_a, token_id_b), ... ] }
# Replace these with real token-ID n-grams from your activation snapshot.
neuron_to_triggers = {
    0: [(1,), (1,2), (1,2,3)],
    1: [(2,), (2,3)],
}

# Deduplicate all n-grams across neurons
unique_ngrams = sorted({ng for lst in neuron_to_triggers.values() for ng in lst})
unique_ngrams



## Batch Counting Across Checkpoints

We query each n‑gram **once** to get all its positions, then compute cumulative counts for **all checkpoints** via a vectorized prefix (`searchsorted` on end‑positions).


In [ ]:

counts_map = batch_counts(unique_ngrams, cutoffs, index, batch_size=2048)
# Inspect a couple
for ng in unique_ngrams[:3]:
    arr = counts_map[ng]
    print(ng, arr[:5], "...", arr[-3:])



## Exposure Feature

For neuron *j* at checkpoint *t*:
\[
\text{exposure}_j(t) = \log\Big( \max_{g \in \text{Top-}k\_j} \text{count}(g; \le T_t) + \epsilon \Big).
\]
Set \(\epsilon = 1\) to avoid \(\log 0\). We assume your `neuron_to_triggers` already represents the top‑k triggers.


In [ ]:

def compute_exposures(neuron_to_triggers, counts_map, epsilon=1.0):
    """Return dict[neuron_id] -> np.ndarray[float] of exposure across checkpoints."""
    exposures = {}
    for j, ngrams in neuron_to_triggers.items():
        if not ngrams:
            exposures[j] = np.zeros_like(next(iter(counts_map.values())), dtype=float)
            continue
        # stack counts for this neuron's triggers
        stacked = np.stack([counts_map[ng] for ng in ngrams], axis=0)  # shape: (K, C)
        mx = stacked.max(axis=0)
        exposures[j] = np.log(mx.astype(np.float64) + float(epsilon))
    return exposures

exposures = compute_exposures(neuron_to_triggers, counts_map, epsilon=1.0)
for j, arr in exposures.items():
    print(f"neuron {j}: ", arr[:5], "...", arr[-3:])



## Quick Plot

Plot exposure over checkpoints for a couple neurons. (Using `matplotlib`, one figure per chart.)


In [ ]:

# Choose a few neurons to visualize
neurons_to_plot = list(exposures.keys())[:2]

for j in neurons_to_plot:
    plt.figure()
    y = exposures[j]
    x = steps
    plt.plot(x, y)
    plt.xlabel("Checkpoint step")
    plt.ylabel("Exposure")
    plt.title(f"Neuron {j} exposure vs. checkpoint")
    plt.show()



## (Optional) Stratified n‑gram Sampling by Frequency

For polytope analysis, you may want 1,000 n‑grams per frequency bin from a **held‑out corpus**. This scaffold assumes you already have a **candidate list** of n‑grams from that corpus and just shows how to bin by **cumulative counts at a chosen cutoff**.


In [ ]:

def stratified_sample_by_frequency(candidate_ngrams, counts_map, cutoff_index, num_bins=10, per_bin=1000, random_state=0):
    """
    Args:
        candidate_ngrams: iterable of token-ID n-grams (present in counts_map)
        counts_map: dict[ngram] -> counts per checkpoint (np.ndarray)
        cutoff_index: which checkpoint index to use for binning
        num_bins: number of frequency bins
        per_bin: target samples per bin
    Returns:
        dict[bin_id] -> list of n-grams
    """
    rng = np.random.RandomState(random_state)
    # Collect (ngram, count_at_cutoff)
    pairs = []
    for ng in candidate_ngrams:
        arr = counts_map.get(ng)
        if arr is not None and cutoff_index < arr.size:
            pairs.append((ng, int(arr[cutoff_index])))
    if not pairs:
        return {i: [] for i in range(num_bins)}
    counts = np.array([c for _, c in pairs], dtype=np.int64)
    # Log-space bins by default
    counts_safe = counts + 1
    edges = np.quantile(np.log1p(counts_safe), np.linspace(0, 1, num_bins + 1))
    # Assign bins
    bin_ids = np.digitize(np.log1p(counts_safe), edges[1:-1], right=False)
    buckets = {i: [] for i in range(num_bins)}
    for (ng, _), b in zip(pairs, bin_ids):
        buckets[b].append(ng)
    # Sample per bin
    sampled = {}
    for b, lst in buckets.items():
        if len(lst) <= per_bin:
            sampled[b] = lst
        else:
            idx = rng.choice(len(lst), size=per_bin, replace=False)
            sampled[b] = [lst[i] for i in idx]
    return sampled

# Demo using our unique_ngrams as "candidates" and the last checkpoint
cutoff_idx = len(steps) - 1
sampled = stratified_sample_by_frequency(unique_ngrams, counts_map, cutoff_idx, num_bins=5, per_bin=5)
for b, lst in sampled.items():
    print(b, lst)



## Notes for OLMo/OLMo‑2

- Swap in the **OLMo tokenizer** and reconstruct the OLMo training stream order.
- Build the same kind of index (Tokengrams or Infini‑gram) over **OLMo token IDs**.
- Use OLMo checkpoint logs to compute the cutoff array `cutoffs_tokens_olmo`; then call `cutoffs = cutoffs_tokens_olmo - 1` before `batch_counts`.
- The rest of the notebook (batch counting, exposure computation, sampling) remains unchanged.
